In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
RANDOM_SEED = 0
# ---------------------------
# Baseline samplers
# ---------------------------
from scipy.stats import qmc
def sample_random(n: int, seed: int) -> np.ndarray:
    r = np.random.RandomState(seed)
    return DOMAIN[0] + r.rand(n,1) * (DOMAIN[1] - DOMAIN[0])

def sample_sobol(n: int, seed: int) -> np.ndarray:
    eng = qmc.Sobol(d=1, scramble=True, seed=seed)
    m = int(np.ceil(np.log2(max((2,n)))))
    pts = eng.random_base2(m=m)[:n,:]
    return DOMAIN[0] + pts * (DOMAIN[1] - DOMAIN[0])

def sample_lhs(n: int, seed: int) -> np.ndarray:
    eng = qmc.LatinHypercube(d=1, seed=seed)
    pts = eng.random(n)
    return DOMAIN[0] + pts * (DOMAIN[1] - DOMAIN[0])

def sample_grid(n: int, seed: int):
    return np.linspace(DOMAIN[0], DOMAIN[1], n).reshape((-1,1))

# ---------------------------
# Bayesian EI sampler
# ---------------------------
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from scipy.stats import norm

def fit_gp_fast(X, y):
    kernel = C(1.0)*RBF(length_scale=0.25) + WhiteKernel(noise_level=1e-6)
    gp = GaussianProcessRegressor(kernel=kernel, optimizer=None, alpha=0.0, normalize_y=True, random_state=0)
    gp.fit(X, y.ravel())
    return gp

def expected_improvement(Xgrid, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(Xcand, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    imp = y_best - mu - xi
    Z = imp / sigma
    return imp * norm.cdf(Z) + sigma * norm.pdf(Z)

def sample_bayesian_ei(n: int, seed: int, init_n: int = 4) -> np.ndarray:
    X = sample_sobol(init_n, seed); y = f(X.ravel())[:,None]
    gp = fit_gp_fast(X, y)
    grid = np.linspace(DOMAIN[0], DOMAIN[1], DENSE_BUDGET).reshape(-1,1)
    while X.shape[0] < n:
        ei = expected_improvement(grid, gp, np.min(y), xi=0.01)
        idx = int(np.argmax(ei))
        x_next = float(grid[idx,0])
        if np.any(np.isclose(X.ravel(), x_next, atol=1e-12)):
            x_next = float((x_next + np.random.uniform(-1e-3, 1e-3)) % DOMAIN[1])
        X = np.vstack([X, [[x_next]]])
        y = np.vstack([y, [[f(np.array([x_next]))[0]]]])
        gp = fit_gp_fast(X, y)
    return X

# ---------------------------
# Models: GP, Improved MLP (Fourier ensemble), RF
# ---------------------------
import torch
import torch.nn as nn

from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

def fit_rf(X, y):
    rf = RandomForestRegressor(n_estimators=350, random_state=RANDOM_SEED)
    rf.fit(X, y.ravel()); return rf

def fourier_features(x_raw: np.ndarray, K: int = 12) -> np.ndarray:
    a, b = DOMAIN
    x = (x_raw - a) / (b - a)
    feats = [x, x**2, x**3, x**4]
    for k in range(1, K+1):
        feats.append(np.sin(2*np.pi*k*x))
        feats.append(np.cos(2*np.pi*k*x))
    return np.column_stack(feats)

# --- 4. Define the MLP Model ---
class TorchMLP(nn.Module):
    def __init__(self):
        super(TorchMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

class ImprovedMLP:
    def __init__(self, K=12, n_ensemble=5, seed=RANDOM_SEED):
        self.K = K
        self.n_ens = n_ensemble
        self.seed = seed
        self.scaler_X = StandardScaler(with_mean=False)
        self.scaler_y = StandardScaler()
        self.models = []

    def fit(self, X, y):
        Phi = fourier_features(X, self.K)
        self.scaler_X.fit(Phi)
        Phi_s = self.scaler_X.transform(Phi)
        y_s = self.scaler_y.fit_transform(y.reshape(-1,1)).ravel()
        self.models = []
        for i in range(self.n_ens):
            mlp = MLPRegressor(hidden_layer_sizes=(256,256,256), activation="relu",
                               solver="adam", learning_rate_init=1.5e-3, alpha=1e-4,
                               max_iter=9000, early_stopping=True, n_iter_no_change=500,
                               validation_fraction=0.2, random_state=self.seed + i)
            mlp.fit(Phi_s, y_s)
            self.models.append(mlp)
        return self

    def predict(self, Xq, return_std=False):
        Phi_q = fourier_features(Xq, self.K)
        Phi_qs = self.scaler_X.transform(Phi_q)
        preds = np.column_stack([m.predict(Phi_qs) for m in self.models])
        preds = self.scaler_y.inverse_transform(preds)
        yq_m = preds.mean(axis=1, keepdims=True)
        yq_s = preds.std(axis=1, keepdims=True)
        
        yqm = yq_m.ravel() # self.scaler_y.inverse_transform(yq_m).ravel()
        yqs = yq_s.ravel() #self.scaler_y.inverse_transform(yq_s).ravel()
        
        if return_std:
            return yqm, yqs
        else:
            return yqm
    
# --- 4. Define the MLP Model and Wrapper ---
class ForresterMLP(nn.Module):
    def __init__(self):
        super(ForresterMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        return self.network(x)

class MLPWrapper:
    def __init__(self, num_epochs=5000, lr=0.001):
        self.model = ForresterMLP()
        self.num_epochs = num_epochs
        self.lr = lr

    def fit(self, x_train, y_train):
        """Trains the MLP model on the provided data."""
        x_train_tensor = torch.from_numpy(x_train).float().view(-1, 1)
        y_train_tensor = torch.from_numpy(y_train).float().view(-1, 1)
        
        loss_function = nn.MSELoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

        self.model.train()
        for epoch in range(self.num_epochs):
            y_pred_train = self.model(x_train_tensor)
            loss = loss_function(y_pred_train, y_train_tensor)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        return self

    def predict(self, x_test):
        """Makes predictions on new data."""
        x_test_tensor = torch.from_numpy(x_test).float().view(-1, 1)
        self.model.eval()
        with torch.no_grad():
            return self.model(x_test_tensor).numpy().flatten()


def fit_mlp(x_train, y_train):
    """A simple wrapper function to fit the MLP model."""
    return MLPWrapper().fit(x_train, y_train)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from pyomo.environ import ConcreteModel, Var, value

from abc import ABC, abstractmethod
from pyomo.environ import Param, RangeSet
from pyomo.environ import ConcreteModel, Var, sin as pysin, sqrt as pysqrt, exp as pyexp
from pyomo.environ import cos as pycos
from pyomo.contrib.mcpp.pyomo_mcpp import McCormick

class Function2D(ABC):
    def __init__(self, envelope_samples=10):
        self.envelope_samples = envelope_samples

    @abstractmethod
    def f(self, x):
        pass

    @abstractmethod
    def f_mccormick(self, a1, b1, a2, b2):
        pass

    def envelope_interval(self, a1, b1, a2, b2):
        """
        Estimate McCormick envelope over [a1, b1] x [a2, b2] using a regular grid
        """
        mc, m = self.f_mccormick(a1, b1, a2, b2)
        x1_samples = np.linspace(a1, b1, self.envelope_samples)
        x2_samples = np.linspace(a2, b2, self.envelope_samples)
        xx1, xx2 = np.meshgrid(x1_samples, x2_samples)
        gap = []
        lower = []
        upper = []
        x_samples = []
        for xi1, xi2 in zip(xx1.flatten(), xx2.flatten()):
            m.x1.set_value(xi1)
            m.x2.set_value(xi2)
            mc.changePoint(m.x1, xi1)
            mc.changePoint(m.x2, xi2)
            luv = mc.concave()
            llo = mc.convex()
            upper.append(luv)
            lower.append(llo)
            gap.append(luv - llo)
            x_samples.append([xi1, xi2])
        return np.array(x_samples), np.array(lower), np.array(upper), np.array(gap)

    def envelope_interval_points(self, a1, b1, a2, b2, xs):
        """
        Estimate McCormick envelope at specified xs points, where xs is (N, 2)
        """
        mc, m = self.f_mccormick(a1, b1, a2, b2)
        def get_upper_lower(m, mc, xi):
            m.x1.set_value(xi[0])
            m.x2.set_value(xi[1])
            mc.changePoint(m.x1, xi[0])
            mc.changePoint(m.x2, xi[1])
            return mc.concave(), mc.convex()
        upper_lower = np.array([get_upper_lower(m, mc, xi) for xi in xs])
        gap = upper_lower[:, 0] - upper_lower[:, 1]
        return xs, upper_lower[:, 1], upper_lower[:, 0], gap


class Langermann2D(Function2D):
    def __init__(self):
        super().__init__()
        self.a = np.array([3.0, 5.0, 2.0, 1.0, 7.0])
        self.c = np.array([1.0, 2.0, 5.0, 2.0, 3.0])

    def f(self, x):
        x = np.asarray(x, dtype=float)
        x1, x2 = x[..., 0], x[..., 1]
        s = (x1[..., None] - self.a) ** 2 + (x2[..., None] - self.a) ** 2
        terms = self.c * np.exp(-s / np.pi) * np.cos(np.pi * s)
        y = terms.sum(axis=-1)
        return y.item() if np.ndim(y) == 0 else y

    def f_mccormick(self, a1, b1, a2, b2):
        EPS = 1e-8
        m = ConcreteModel()
        m.I = RangeSet(len(self.a))
        m.x1 = Var(bounds=(a1, b1))
        m.x2 = Var(bounds=(a2, b2))
        m.A = Param(m.I, initialize={i + 1: float(self.a[i]) for i in range(len(self.a))})
        m.C = Param(m.I, initialize={i + 1: float(self.c[i]) for i in range(len(self.c))})

        # The Pyomo expression for the 2D Langermann sum
        expr = sum(
            m.C[i] *
            pyexp(-((m.x1 - m.A[i])**2 + (m.x2 - m.A[i])**2 + EPS) / float(np.pi)) *
            pycos(float(np.pi) * ((m.x1 - m.A[i])**2 + (m.x2 - m.A[i])**2 + EPS))
            for i in m.I
        )
        mc = McCormick(expr)
        return mc, m

class Schwefel2D(Function2D):
    def __init__(self):
        super().__init__()
        self.constant = 418.9829
        self.EPS = 1e-8

    def f(self, x):
        x = np.asarray(x, dtype=float)
        x1, x2 = x[..., 0], x[..., 1]
        return self.constant - (x1 * np.sin(np.sqrt(np.abs(x1) + self.EPS)) + x2 * np.sin(np.sqrt(np.abs(x2) + self.EPS)))

    def f_mccormick(self, a1, b1, a2, b2):
        """
        Compute McCormick relaxation for 2D Schwefel function:
            f(x, y) = 418.9829*2 - [x*sin(sqrt(|x|)) + y*sin(sqrt(|y|))]

        with numerical safeguards for abs(0) and sqrt(0).
        """

        m = ConcreteModel()
        m.x1 = Var(bounds=(a1, b1))
        m.x2 = Var(bounds=(a2, b2))

        # Add epsilon inside sqrt(abs(.)) to prevent NaN or undefined derivative at 0
        expr = (
            418.9829 * 2
            - (
                m.x1 * pysin(pysqrt(abs(m.x1) + self.EPS))
                + m.x2 * pysin(pysqrt(abs(m.x2) + self.EPS))
            )
        )

        mc = McCormick(expr)
        return mc, m
    


In [6]:
from functions.function import Forrester, Schwefel, Higdon, GrammacyLee, Sigmoid, Ackley, Langermann, Griewank

from interval_builder import IntervalBuilder
from envelope_builder import piecewise_envelopes
from samplers.adaptive_sampler import AdaptiveSampler

In [7]:
FUNCTIONS = {
    "Ackley": {"domain": [-32.768, 32.768], 'f': Ackley},
    "Schwefel": {'domain':[-500, 500], 'f': Schwefel},
    "Sigmoid": {'domain': [-6, 6], 'f': Sigmoid},
    "GramacyLee": {'domain':[0.5, 2.5], 'f': GrammacyLee},
    "Higdon": {'domain': [0, 10], 'f': Higdon},
    "Forrester": {'domain': [0, 2.5], 'f': Forrester},
    "Langermann": {'domain': [-10, 20], 'f': Langermann},
    "Griewank": {'domain': [-50, 50], 'f': Griewank},
}

selection = 'Schwefel'
OVERALL_BUDGET = 200
sampling_budget = 0.2

BUDGET = int(OVERALL_BUDGET * sampling_budget)
REFINING = int(OVERALL_BUDGET * (1 - sampling_budget))

# BUDGET = 100
GRID_EVAL = 10

func = f2d #FUNCTIONS[selection]['f']()
DOMAIN = FUNCTIONS[selection]['domain']
# DOMAIN[0] *= 2
# DOMAIN[1] *= 2
# DOMAIN = (-30.0, 0.0)
# Minimum samples per gap
DOMAIN_SPLITS_PC = 0.3
DOMAIN_SPLITS = 10 #int(np.ceil(1/DOMAIN_SPLITS_PC))

MIN_SAMPLE_PC = 0.01
MIN_SAMPLES = int(BUDGET * MIN_SAMPLE_PC)
MIN_SAMPLES = MIN_SAMPLES if MIN_SAMPLES else 1
MIN_SAMPLES = 1

CONFIG = {
    "n_samples": 15,
#     "max_envelope_size": 0.2,
    "max_envelope_size": 1.0,
    'max_gap': 0.5,
    'seed': 42,
    'intervals': {
        'a': DOMAIN[0],
        'b': DOMAIN[1],
    },
    "budget": BUDGET,
}
f = f2d.f

In [41]:
OVERALL_BUDGET = 200
sampling_budget = 0.6

BUDGET = OVERALL_BUDGET * sampling_budget
REFINING = OVERALL_BUDGET * (1 - sampling_budget)

def method0_uncertanity2d(func, sample_coarse=True, return_domains=True, mccormick_sampling=False):
    sampler = AdaptiveSampler(func, BUDGET, False)
    breakpoints = np.linspace(DOMAIN[0], DOMAIN[1], DOMAIN_SPLITS + 1)
#     Xgrid = np.linspace(DOMAIN[0], DOMAIN[1], DOMAIN_SPLITS)
#     Ygrid = func.f(Xgrid)

#     breakpoints = [DOMAIN[0], DOMAIN[1]]

    domains = []
    gaps = []
    for i in range(len(breakpoints) - 1):
        for j in range(len(breakpoints) - 1):
            a1, b1 = breakpoints[i], breakpoints[i + 1]
            a2, b2 = breakpoints[j], breakpoints[j + 1]
            print(a1,b1,a2,b2) 
            mc, m = func.f_mccormick(a1, b1, a2, b2)
            if sample_coarse:
                lb, ub = compute_bounds_on_expr(mc.pyomo_expr)
                gap = [ub - lb]
            else:
                _, _, _, gap = func.envelope_interval(a1, b1, a2, b2)
            domains.append((a1, b1, a2, b2))
            gaps.append(max(gap))
        
    sample_dist = allocate_samples(gaps, BUDGET, MIN_SAMPLES)
    print('Sample Allocations: ', sample_dist)
    sampled_x = []
    for idx in range(len(domains)):
        a1,b1,a2,b2 = domains[idx]
        s = sample_dist[idx]
        if mccormick_sampling:
            samples, _ = sampler.sample_points(domain=domains[idx], points_n=s)
        else:
            if idx > 0:
                xss = np.linspace(a1, b1, s + 1)[1:]
                yss = np.linspace(a2, b2, s+ 1)[1:]
                samples = list(zip(xss, yss))
            else:
                xss = np.linspace(a1, b1, s)
                yss = np.linspace(a2, b2, s)
                samples = list(zip(xss, yss))
        sampled_x.extend(samples)
    sampled_x.sort()
    sampled_x = np.array(sampled_x)
    sampled_y = func.f(sampled_x)
    
    if return_domains:
        return sampled_x, sampled_y, domains, gaps
    else:
        return sampled_x, sampled_y

from typing import List

def allocate_samples(weights: List[float], total_samples: int, min_per_gap: int) -> List[int]:
    """
    Allocate total_samples across gaps proportionally to weights,
    ensuring at least min_per_gap samples for each gap.
    
    Args:
        weights (List[float]): List of weights (e.g., gap sizes).
        total_samples (int): Total number of samples to allocate.
        min_per_gap (int): Minimum samples each gap must receive.
        
    Returns:
        List[int]: Final allocation of samples per gap.
    """
    n = len(weights)
    # Step 1: Assign minimum to each gap
    base_alloc = [min_per_gap] * n
    remaining = total_samples - n * min_per_gap
    
    if remaining < 0:
        raise ValueError("Not enough samples to satisfy minimum requirement for all gaps.")
    
    # Step 2: Normalize weights
    total_weight = sum(weights)
    proportions = [w / total_weight for w in weights]
    
    # Step 3: Proportional allocation
    proportional_alloc = [remaining * p for p in proportions]
    
    # Step 4: Combine and round
    final_alloc = [base + int(round(prop)) for base, prop in zip(base_alloc, proportional_alloc)]
    
    # Step 5: Adjust to ensure sum matches total_samples
    diff = total_samples - sum(final_alloc)
    
    # Distribute remainder (if rounding caused mismatch)
    i = 0
    while diff != 0:
        if diff > 0:
            final_alloc[i % n] += 1
            diff -= 1
        else:
            if final_alloc[i % n] > min_per_gap:
                final_alloc[i % n] -= 1
                diff += 1
        i += 1
    
    return final_alloc

import bisect

def assign_domains(points, domains):
    boundaries = [d[1] for d in domains]
    result = []
    for p in points:
        idx = bisect.bisect_left(boundaries, p)
        if idx == 0:
            d = domains[0]
            if d[0] <= p <= d[1]:
                result.append(idx)
            else:
                result.append(-1)
        elif idx < len(domains):
            d = domains[idx]
            if d[0] < p <= d[1]:
                result.append(idx)
            else:
                result.append(-1)
        else:
            result.append(-1)
    return np.array(result)


def assign_domains_2d(points, domains):
    result = []
    for p in points:
        found = -1
        for idx, d in enumerate(domains):
            x_low, x_high = d[0]
            y_low, y_high = d[1]
            if x_low <= p[0] <= x_high and y_low <= p[1] <= y_high:
                found = idx
                break
        result.append(found)
    return np.array(result)





In [42]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

def fit_gp_2d(X, y):
    # Solid default kernel: Matern + Constant + low-noise white kernel
    kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(1e-6)
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3, random_state=0)
    gp.fit(X, y)
    return gp

In [43]:
from scipy.stats import norm

def expected_improvement(mu, std, y_opt=0.0, xi=0.01):
    values = np.zeros_like(mu, dtype=np.float64)
    mask = std > 0
    improve = y_opt - xi - mu[mask]
    scaled = improve / std[mask]
    cdf = norm.cdf(scaled)
    pdf = norm.pdf(scaled)
    exploit = improve * cdf
    explore = std[mask] * pdf
    values[mask] = exploit + explore

    return values

In [44]:
# DOMAIN is still [lo, hi] scalars — same as 1D.
# For 2D we treat both axes as having the same domain bounds,
# i.e. the domain is [lo, hi] x [lo, hi].

def sample_random_2d(n: int, seed: int) -> np.ndarray:
    r = np.random.RandomState(seed)
    pts = r.rand(n, 2)
    return DOMAIN[0] + pts * (DOMAIN[1] - DOMAIN[0])

def sample_sobol_2d(n: int, seed: int) -> np.ndarray:
    eng = qmc.Sobol(d=2, scramble=True, seed=seed)
    m = int(np.ceil(np.log2(max(2, n))))
    pts = eng.random_base2(m=m)[:n, :]          # (n, 2) in [0,1)^2
    return DOMAIN[0] + pts * (DOMAIN[1] - DOMAIN[0])

def sample_lhs_2d(n: int, seed: int) -> np.ndarray:
    eng = qmc.LatinHypercube(d=2, seed=seed)
    pts = eng.random(n)                         # (n, 2) in [0,1)^2
    return DOMAIN[0] + pts * (DOMAIN[1] - DOMAIN[0])

def sample_grid_2d(n: int, seed: int) -> np.ndarray:
    k = int(np.ceil(np.sqrt(n)))
    g = np.linspace(DOMAIN[0], DOMAIN[1], k)   # same bounds on both axes
    xx1, xx2 = np.meshgrid(g, g)
    return np.column_stack([xx1.ravel(), xx2.ravel()])


def method_0a_2d(func: Function2D,
                 sample_coarse: bool = False,
                 return_domains: bool = False):

    def compute_gap(a1, b1, a2, b2):
        mc, m = func.f_mccormick(a1, b1, a2, b2)
        if sample_coarse:
            lb, ub = compute_bounds_on_expr(mc.pyomo_expr)
            return float(ub - lb)
        else:
            _, _, _, gap = func.envelope_interval(a1, b1, a2, b2)
            return float(max(gap))

    # Single starting rectangle: [lo,hi] x [lo,hi]
    lo, hi = float(DOMAIN[0]), float(DOMAIN[1])

    domains: list = [(lo, hi, lo, hi)]
    gaps:    list = [compute_gap(lo, hi, lo, hi)]

    target = BUDGET - 3   # net +3 per split (1 → 4 children)

    while len(domains) < target:
        domain_idx = int(np.argmax(gaps))
        a1, b1, a2, b2 = domains.pop(domain_idx)
        gaps.pop(domain_idx)

        mid1 = (a1 + b1) / 2.0
        mid2 = (a2 + b2) / 2.0

        children = [
            (a1,   mid1, a2,   mid2),
            (mid1, b1,   a2,   mid2),
            (a1,   mid1, mid2, b2  ),
            (mid1, b1,   mid2, b2  ),
        ]

        for child in children:
            domains.append(child)
            gaps.append(compute_gap(*child))

    # Collect unique corners from all surviving rectangles
    corners = []
    for (a1, b1, a2, b2) in domains:
        corners.extend([[a1, a2], [a1, b2], [b1, a2], [b1, b2]])

    sampled_x = np.unique(np.array(corners), axis=0)
    sampled_x = sampled_x[np.lexsort((sampled_x[:, 1], sampled_x[:, 0]))]
    sampled_y = func.f(sampled_x)

    if return_domains:
        return sampled_x, sampled_y, domains, gaps
    return sampled_x, sampled_y



In [45]:
import os
from tqdm import tqdm
from sklearn.metrics import mean_absolute_percentage_error
from typing import List, Tuple, Callable, Dict

In [19]:
rmse: Dict[Tuple[str,str], float] = {}
mape: Dict[Tuple[str,str], float] = {}
samples: Dict[str, np.ndarray] = {}  # name -> (n,2)
preds:   Dict[Tuple[str,str], np.ndarray] = {}
stds:   Dict[Tuple[str,str], np.ndarray] = {}

In [ ]:
X, y, domains, gaps = method0_uncertanity2d(func, sample_coarse=False, mccormick_sampling=False)

X_og = X.copy()
y_og = y.copy()

# X = X.reshape((-1, 1))
# y = y.reshape((-1, 1))
f = func.f


n_test = 100
x1_test = np.linspace(DOMAIN[0], DOMAIN[1], n_test)
x2_test = np.linspace(DOMAIN[0], DOMAIN[1], n_test)
xx1, xx2 = np.meshgrid(x1_test, x2_test)
X_test = np.column_stack([xx1.ravel(), xx2.ravel()])
# X_test = np.linspace(DOMAIN[0], DOMAIN[1], 1500)
y_test = f(X_test)

sample_dist = allocate_samples(gaps, 500, 1)
mask = np.array(sample_dist) > 2
final_mask = np.zeros(X_test.shape[0], dtype=bool)

for d in np.array(domains)[mask]:
    # d = [x1_min, x2_min, x1_max, x2_max]
    in_domain = (
        (X_test[:, 0] >= d[0]) & (X_test[:, 0] <= d[2]) &
        (X_test[:, 1] >= d[1]) & (X_test[:, 1] <= d[3])
    )
    final_mask |= in_domain

X_test = X_test[final_mask] if sum(final_mask) else X_test
y_test = f(X_test)

n_test = 500
x1_test = np.linspace(DOMAIN[0], DOMAIN[1], n_test)
x2_test = np.linspace(DOMAIN[0], DOMAIN[1], n_test)
xx1, xx2 = np.meshgrid(x1_test, x2_test)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel()])
# X_grid = np.linspace(DOMAIN[0], DOMAIN[1], 1000)
y_grid_true = f(X_grid)

# L, U = piecewise_envelopes(func, X_grid, y_grid_true, np.linspace(DOMAIN[0], DOMAIN[1], 11))
# plt.plot(X_grid, L, c='orange')
# plt.plot(X_grid, U, c='green')
# plt.scatter(X, y, s=25, marker="x", label="Samples")
# plt.show()

X_domains = assign_domains_2d(X_grid, np.array(domains).reshape((-1,2,2)))
# Weights by domains
X_grid_gaps = np.array(gaps)[X_domains.ravel()]
domain_weights = (X_grid_gaps / sum(gaps)) * 100


n_add = REFINING
# add_per_it = 5
xi = 0.01

# resampling grid based
# X = np.linspace(DOMAIN[0], DOMAIN[1], BUDGET)
# X = X.reshape((-1,1))
# y = f(X)

np.random.seed(RANDOM_SEED)

strategies: List[Tuple[str, Callable[[int,int], np.ndarray]]] = [
    ("Sobol",              sample_sobol_2d),
    ("LHS",                sample_lhs_2d),
    ("Grid",             sample_grid_2d),
    ("Approach B", lambda n, seed: method_0a_2d(func)),
    ("Approach A - Uniform Sampling", lambda n, seed: method0_uncertanity2d(func, sample_coarse=False, return_domains=False, mccormick_sampling=False)),
]

# history = []

outputs = {}

rmse: Dict[Tuple[str,str], float] = {}
mape: Dict[Tuple[str,str], float] = {}
samples: Dict[str, np.ndarray] = {}  # name -> (n,2)
preds:   Dict[Tuple[str,str], np.ndarray] = {}
stds:   Dict[Tuple[str,str], np.ndarray] = {}


for strategy, gen in strategies:
    BUDGET = 300
    print('Strategy : ', strategy, BUDGET)
    if strategy in ['Sobol', 'LHS', 'Grid']:
        X = gen(BUDGET, RANDOM_SEED)
        y = func.f(X)
    else:
        X, y = gen(BUDGET, RANDOM_SEED)
    
    for surrogate in ['NN', 'GP']:
        if surrogate != 'NN':
            ensemble = fit_gp_2d(X, y)
        else:
            ensemble = ImprovedMLP(K=2, n_ensemble=5)
            ensemble.fit(X, y)

        mu, std = ensemble.predict(X_test, return_std=True)
        yhat = mu
        ystd = std

        name = strategy
        mname = surrogate

        preds[(name, mname)] = (yhat, ystd)
        rmse[(name, mname)] = float(np.sqrt(mean_squared_error(y_test, np.array(yhat).ravel())))
        mape[(name, mname)] = float(mean_absolute_percentage_error(y_test, np.array(yhat).ravel()))
    
    samples[name] = (X.copy(),y.copy())
    outputs[name] = (preds, rmse, mape, samples)

-500.0 -400.0 -500.0 -400.0
-500.0 -400.0 -400.0 -300.0
-500.0 -400.0 -300.0 -200.0
-500.0 -400.0 -200.0 -100.0
-500.0 -400.0 -100.0 0.0
-500.0 -400.0 0.0 100.0
-500.0 -400.0 100.0 200.0
-500.0 -400.0 200.0 300.0
-500.0 -400.0 300.0 400.0
-500.0 -400.0 400.0 500.0
-400.0 -300.0 -500.0 -400.0
-400.0 -300.0 -400.0 -300.0
-400.0 -300.0 -300.0 -200.0
-400.0 -300.0 -200.0 -100.0
-400.0 -300.0 -100.0 0.0
-400.0 -300.0 0.0 100.0
-400.0 -300.0 100.0 200.0
-400.0 -300.0 200.0 300.0
-400.0 -300.0 300.0 400.0
-400.0 -300.0 400.0 500.0
-300.0 -200.0 -500.0 -400.0
-300.0 -200.0 -400.0 -300.0
-300.0 -200.0 -300.0 -200.0
-300.0 -200.0 -200.0 -100.0
-300.0 -200.0 -100.0 0.0
-300.0 -200.0 0.0 100.0
-300.0 -200.0 100.0 200.0
-300.0 -200.0 200.0 300.0
-300.0 -200.0 300.0 400.0
-300.0 -200.0 400.0 500.0
-200.0 -100.0 -500.0 -400.0
-200.0 -100.0 -400.0 -300.0
-200.0 -100.0 -300.0 -200.0
-200.0 -100.0 -200.0 -100.0
-200.0 -100.0 -100.0 0.0
-200.0 -100.0 0.0 100.0
-200.0 -100.0 100.0 200.0
-200.0 -100.0 200.

/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.



Strategy :  LHS 300


/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.



Strategy :  Grid 300
Strategy :  Approach B 300
Strategy :  Approach A - Uniform Sampling 300
-500.0 -400.0 -500.0 -400.0
-500.0 -400.0 -400.0 -300.0
-500.0 -400.0 -300.0 -200.0
-500.0 -400.0 -200.0 -100.0
-500.0 -400.0 -100.0 0.0
-500.0 -400.0 0.0 100.0
-500.0 -400.0 100.0 200.0
-500.0 -400.0 200.0 300.0
-500.0 -400.0 300.0 400.0
-500.0 -400.0 400.0 500.0
-400.0 -300.0 -500.0 -400.0
-400.0 -300.0 -400.0 -300.0
-400.0 -300.0 -300.0 -200.0
-400.0 -300.0 -200.0 -100.0
-400.0 -300.0 -100.0 0.0
-400.0 -300.0 0.0 100.0
-400.0 -300.0 100.0 200.0
-400.0 -300.0 200.0 300.0
-400.0 -300.0 300.0 400.0
-400.0 -300.0 400.0 500.0
-300.0 -200.0 -500.0 -400.0
-300.0 -200.0 -400.0 -300.0
-300.0 -200.0 -300.0 -200.0
-300.0 -200.0 -200.0 -100.0
-300.0 -200.0 -100.0 0.0
-300.0 -200.0 0.0 100.0
-300.0 -200.0 100.0 200.0
-300.0 -200.0 200.0 300.0
-300.0 -200.0 300.0 400.0
-300.0 -200.0 400.0 500.0
-200.0 -100.0 -500.0 -400.0
-200.0 -100.0 -400.0 -300.0
-200.0 -100.0 -300.0 -200.0
-200.0 -100.0 -200.0 -100.0

/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.



In [ ]:
surrogate = 'GP'
experiment_name = '{}+STD + warm_start - {}'.format(surrogate, BUDGET)

output_dir = './plots/2D/it1/{}'.format(experiment_name)

output_dir_plot = '{}/sampling/'.format(output_dir)
output_dir_ei = '{}/ei/'.format(output_dir)
os.makedirs(output_dir_plot, exist_ok=True)
os.makedirs(output_dir_ei, exist_ok=True)


OVERALL_BUDGET = 300
sampling_budget = 0.6

BUDGET = OVERALL_BUDGET * sampling_budget
REFINING = OVERALL_BUDGET * (1 - sampling_budget)


X, y, domains, gaps = method0_uncertanity2d(func, sample_coarse=False, mccormick_sampling=False)

X_og = X.copy()
y_og = y.copy()

# X = X.reshape((-1, 1))
# y = y.reshape((-1, 1))
f = func.f


n_test = 100
x1_test = np.linspace(DOMAIN[0], DOMAIN[1], n_test)
x2_test = np.linspace(DOMAIN[0], DOMAIN[1], n_test)
xx1, xx2 = np.meshgrid(x1_test, x2_test)
X_test = np.column_stack([xx1.ravel(), xx2.ravel()])
# X_test = np.linspace(DOMAIN[0], DOMAIN[1], 1500)
y_test = f(X_test)

sample_dist = allocate_samples(gaps, 200, 1)
mask = np.array(sample_dist) > 2
final_mask = np.zeros(X_test.shape[0], dtype=bool)

for d in np.array(domains)[mask]:
    # d = [x1_min, x2_min, x1_max, x2_max]
    in_domain = (
        (X_test[:, 0] >= d[0]) & (X_test[:, 0] <= d[2]) &
        (X_test[:, 1] >= d[1]) & (X_test[:, 1] <= d[3])
    )
    final_mask |= in_domain

X_test = X_test[final_mask] if sum(final_mask) else X_test
y_test = f(X_test)

n_test = 300
x1_test = np.linspace(DOMAIN[0], DOMAIN[1], n_test)
x2_test = np.linspace(DOMAIN[0], DOMAIN[1], n_test)
xx1, xx2 = np.meshgrid(x1_test, x2_test)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel()])
y_grid_true = f(X_grid)


X_domains = assign_domains_2d(X_grid, np.array(domains).reshape((-1,2,2)))
# Weights by domains
X_grid_gaps = np.array(gaps)[X_domains.ravel()]
domain_weights = (X_grid_gaps / sum(gaps)) * 100

n_add = REFINING
# add_per_it = 5
xi = 0.01
ensemble = ImprovedMLP(K=2, n_ensemble=5)



strategies = [
    ('EI', lambda mu,std, f_best, xi: expected_improvement(mu, std, f_best, xi=xi)),
    ('EI (Std * domain_weights) ', lambda mu,std, f_best, xi: expected_improvement(mu, std * domain_weights, f_best, xi=xi)),
    ('STD', lambda mu,std, f_best, xi: std ),
    ('STD * domain_weights', lambda mu,std, f_best, xi: std * domain_weights),
]


for strategy, ei_func in strategies:
    print('Strategy : ', strategy)
    X = X_og.copy()
    y = y_og.copy()

    
    for t in tqdm(range(int(n_add))):
        if surrogate != 'NN':
            ensemble = fit_gp_2d(X, y)
        else:
            ensemble.fit(X, y)

        mu, std = ensemble.predict(X_grid, return_std=True)

        f_best = np.min(y)
        ei = ei_func(mu, std, f_best, xi)
        idx = int(np.argmax(ei))
        x_next = X_grid[idx : idx + 1]

        # avoid duplicates
        tries = 0
        stds = std.copy()
        while np.min(np.abs(X - x_next)) < ((DOMAIN[1] - DOMAIN[0]) * 0.01) and tries < 10:
            ei[idx] = -np.inf
            idx = int(np.argmax(ei))
            x_next = X_grid[idx : idx + 1]
            tries += 1

        y_next = f(x_next)  # 1D length-1    \

        X = np.vstack([X, x_next])
        y = np.concatenate([y, y_next])


    mu, std = ensemble.predict(X_test, return_std=True)
    yhat = mu
    ystd = std

    name = strategy
    mname = surrogate

    preds[(name, mname)] = (yhat, ystd)
    rmse[(name, mname)] = float(np.sqrt(mean_squared_error(y_test, np.array(yhat).ravel())))
    mape[(name, mname)] = float(mean_absolute_percentage_error(y_test, np.array(yhat).ravel()))
    samples[name] = (X.copy(),y.copy())
    
    ensemble = ImprovedMLP(K=5, n_ensemble=5)
    ensemble.fit(X,y)
    mu, std = ensemble.predict(X_grid, return_std=True)


    mu, std = ensemble.predict(X_test, return_std=True)
    yhat = mu
    ystd = std

    name = strategy
    mname = 'NN'

    preds[(name, mname)] = (yhat, ystd)
    rmse[(name, mname)] = float(np.sqrt(mean_squared_error(y_test, np.array(yhat).ravel())))
    mape[(name, mname)] = float(mean_absolute_percentage_error(y_test, np.array(yhat).ravel()))
    
    outputs[name] = (preds, rmse, mape, samples)

-500.0 -400.0 -500.0 -400.0
-500.0 -400.0 -400.0 -300.0
-500.0 -400.0 -300.0 -200.0
-500.0 -400.0 -200.0 -100.0
-500.0 -400.0 -100.0 0.0
-500.0 -400.0 0.0 100.0
-500.0 -400.0 100.0 200.0
-500.0 -400.0 200.0 300.0
-500.0 -400.0 300.0 400.0
-500.0 -400.0 400.0 500.0
-400.0 -300.0 -500.0 -400.0
-400.0 -300.0 -400.0 -300.0
-400.0 -300.0 -300.0 -200.0
-400.0 -300.0 -200.0 -100.0
-400.0 -300.0 -100.0 0.0
-400.0 -300.0 0.0 100.0
-400.0 -300.0 100.0 200.0
-400.0 -300.0 200.0 300.0
-400.0 -300.0 300.0 400.0
-400.0 -300.0 400.0 500.0
-300.0 -200.0 -500.0 -400.0
-300.0 -200.0 -400.0 -300.0
-300.0 -200.0 -300.0 -200.0
-300.0 -200.0 -200.0 -100.0
-300.0 -200.0 -100.0 0.0
-300.0 -200.0 0.0 100.0
-300.0 -200.0 100.0 200.0
-300.0 -200.0 200.0 300.0
-300.0 -200.0 300.0 400.0
-300.0 -200.0 400.0 500.0
-200.0 -100.0 -500.0 -400.0
-200.0 -100.0 -400.0 -300.0
-200.0 -100.0 -300.0 -200.0
-200.0 -100.0 -200.0 -100.0
-200.0 -100.0 -100.0 0.0
-200.0 -100.0 0.0 100.0
-200.0 -100.0 100.0 200.0
-200.0 -100.0 200.

  0%|                                                                                                   | 0/120 [00:00<?, ?it/s]/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

  4%|███▊                                                                                       | 5/120 [00:03<01:16,  1.50it/s]/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

  5%|████▌                                                                                      | 6/120 [00:03<01:15,  1.51it/s]/opt/anaconda3/e

 35%|███████████████████████████████▍                                                          | 42/120 [00:39<01:21,  1.04s/it]/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

 36%|████████████████████████████████▎                                                         | 43/120 [00:40<01:21,  1.05s/it]/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

 37%|█████████████████████████████████                                                         | 44/120 [00:41<01:17,  1.02s/it]/opt/anaconda3/e

 67%|████████████████████████████████████████████████████████████                              | 80/120 [01:14<00:36,  1.09it/s]/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

 68%|████████████████████████████████████████████████████████████▊                             | 81/120 [01:15<00:36,  1.07it/s]/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

 68%|█████████████████████████████████████████████████████████████▌                            | 82/120 [01:16<00:36,  1.03it/s]/opt/anaconda3/e

 98%|██████████████████████████████████████████████████████████████████████████████████████▊  | 117/120 [01:50<00:03,  1.06s/it]/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

 98%|███████████████████████████████████████████████████████████████████████████████████████▌ | 118/120 [01:51<00:02,  1.09s/it]/opt/anaconda3/envs/thesis/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

 99%|████████████████████████████████████████████████████████████████████████████████████████▎| 119/120 [01:52<00:01,  1.06s/it]/opt/anaconda3/e

ValueError: Found array with 0 sample(s) (shape=(0, 2)) while a minimum of 1 is required by GaussianProcessRegressor.

In [103]:
import pandas as pd

In [104]:
output_data = {}
for key, val in samples.items():
    X,y = val
    domain_id, domain_count = np.unique(assign_domains_2d(X, np.array(domains).reshape((-1,2,2))), return_counts=True)
    output_data[key] = (domain_id, domain_count)


    all_domain_ids = sorted(set(np.concatenate([v[0] for v in output_data.values()])))
    records = []

    for domain_id in all_domain_ids:
        x_min, x_max, y_min, y_max = domains[domain_id]

        row = {
            "domain_id": domain_id,
            "x_min": x_min,
            "x_max": x_max,
            "y_min": y_min,
            "y_max": y_max,
        }

        
        for out_name, (ids, counts) in output_data.items():
            if domain_id in ids:
                idx = np.where(ids == domain_id)[0][0]
                row[f"{out_name}"] = counts[idx]
            else:
                row[f"{out_name}"] = 0

        records.append(row)

df = pd.DataFrame(records)
df

,domain_id,x_min,x_max,y_min,y_max,Sobol,LHS,Grid,Approach B,Approach A - Uniform Sampling,EI,EI (Std * domain_weights),STD,STD * domain_weights
0,0,-500.0,-400.0,-500.0,-400.0,2,2,4,7,3,3,4,12,11
1,1,-500.0,-400.0,-400.0,-300.0,3,2,4,6,2,2,3,6,5
2,2,-500.0,-400.0,-300.0,-200.0,3,3,4,5,2,2,2,6,6
3,3,-500.0,-400.0,-200.0,-100.0,3,5,2,4,2,2,3,8,8
4,4,-500.0,-400.0,-100.0,0.0,4,6,4,4,3,3,3,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,400.0,500.0,0.0,100.0,4,0,4,2,3,3,3,5,6
96,96,400.0,500.0,100.0,200.0,2,1,2,5,3,3,3,5,7
97,97,400.0,500.0,200.0,300.0,5,2,4,7,3,3,3,5,6
98,98,400.0,500.0,300.0,400.0,2,9,4,6,3,3,3,5,6


In [101]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "iframe_connected"

fig = go.Figure()

first_orig, first_lower, first_upper = True, True, True

# ----------------------------------------------------------------------
# Plot the function + McCormick envelopes
# ----------------------------------------------------------------------
for domain in domains:
    a1, b1, a2, b2 = domain
    xs, lower, upper, gap = f2d.envelope_interval(a1, b1, a2, b2)
    x1, x2 = xs[:, 0], xs[:, 1]
    orig = np.array([f2d.f(pt) for pt in xs])

    n_samples = int(np.sqrt(xs.shape[0]))
    x1_grid = x1.reshape(n_samples, n_samples)
    x2_grid = x2.reshape(n_samples, n_samples)
    orig_grid = orig.reshape(n_samples, n_samples)
    lower_grid = lower.reshape(n_samples, n_samples)
    upper_grid = upper.reshape(n_samples, n_samples)

    # Original Function
    fig.add_trace(go.Surface(
        x=x1_grid, y=x2_grid, z=orig_grid,
        colorscale='Viridis', opacity=0.7,
        name="Original Function",
        showlegend=first_orig,
        legendgroup='original',
        hoverinfo='skip'
    ))
    first_orig = False

    # Lower Envelope
    fig.add_trace(go.Surface(
        x=x1_grid, y=x2_grid, z=lower_grid,
        colorscale='Blues', opacity=0.6,
        name="Lower Envelope",
        showlegend=first_lower,
        legendgroup='lower',
        hoverinfo='skip'
    ))
    first_lower = False

    # Upper Envelope
    fig.add_trace(go.Surface(
        x=x1_grid, y=x2_grid, z=upper_grid,
        colorscale='Reds', opacity=0.6,
        name="Upper Envelope",
        showlegend=first_upper,
        legendgroup='upper',
        hoverinfo='skip'
    ))
    first_upper = False

# ----------------------------------------------------------------------
# Metric Bars per Domain (grouped + legend toggle fixed)
# ----------------------------------------------------------------------
df["x_center"] = (df["x_min"] + df["x_max"]) / 2
df["y_center"] = (df["y_min"] + df["y_max"]) / 2

metrics = ['EI', 'EI (Std * domain_weights) ', 'STD', 'STD * domain_weights']
colors = ['orange', 'purple', 'green', 'red']

# Compute spacing for bar placement
x_centers_sorted = sorted(df["x_center"].unique())
y_centers_sorted = sorted(df["y_center"].unique())
dx = np.mean(np.diff(x_centers_sorted)) if len(x_centers_sorted) > 1 else 0.05
dy = np.mean(np.diff(y_centers_sorted)) if len(y_centers_sorted) > 1 else 0.05

offsets = [
    (-0.15 * dx,  0.15 * dy),
    ( 0.15 * dx,  0.15 * dy),
    (-0.15 * dx, -0.15 * dy),
    ( 0.15 * dx, -0.15 * dy),
]

# Base Z and scaling
z_base = np.max(upper_grid) * 1.05
z_range = np.ptp(upper_grid)
max_val = max(df[m].max() for m in metrics)
bar_scale = 0.5 #z_range * 1.3 / max_val
line_thickness = 8  # slimmer bars

# --- Add bars grouped per metric ---
for metric, color, (ox, oy) in zip(metrics, colors, offsets):
    first_metric = True  # show legend only for first trace in group

    for _, row in df.iterrows():
        x = row["x_center"] + ox
        y = row["y_center"] + oy
        val = row[metric]
        z0 = z_base
        z1 = z_base + val * bar_scale

        hover_label = (
            f"<b>Domain {row['domain_id']}</b><br>"
            f"{metric}: {val:.4f}<br>"
            f"x_center: {row['x_center']:.3f}<br>"
            f"y_center: {row['y_center']:.3f}"
        )

        fig.add_trace(go.Scatter3d(
            x=[x, x],
            y=[y, y],
            z=[z0, z1],
            mode='lines',
            line=dict(color=color, width=line_thickness),
            hovertext=hover_label,
            hoverinfo='text',
            name=metric if first_metric else None,
            legendgroup=metric,
            showlegend=first_metric
        ))

        first_metric = False  # only first trace shows legend entry

# ----------------------------------------------------------------------
# Layout
# ----------------------------------------------------------------------
fig.update_layout(
    title="Langermann 2D Function + McCormick Envelopes + Toggleable Metric Bars",
    scene=dict(
        xaxis_title='x1',
        yaxis_title='x2',
        zaxis_title='Function Value',
        aspectmode='cube'
    ),
    legend=dict(
        x=0.02, y=0.98,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=11)
    ),
    height=950,
    width=1200
)

fig.show()
